# EEG-to-Image Generation: CSBrain Encoder + FLUX.1-schnell

This notebook implements a multimodal **EEG-to-Image** pipeline where:
- **Encoder**: CSBrain (frozen EEG foundation model) extracts EEG representations
- **Projection**: A 2-layer MLP is trained to align EEG features with **CLIP-L text embedding space** (no paired image data needed!)
- **Generator**: [FLUX.1-schnell](https://huggingface.co/black-forest-labs/FLUX.1-schnell) (Apache 2.0 license) generates 512×512 images

Two generation strategies are demonstrated:
1. **Approach 1 — Text-guided**: Class label → curated visual text prompt → FLUX.1-schnell
2. **Approach 2 — EEG-conditioned**: EEG signal → projected CLIP embedding → injected as `pooled_prompt_embeds` in FLUX.1-schnell

```
EEG (batch, 22, 4, 200)
  → CSBrain (frozen) → (batch, 22, 4, 200)
  → Region pooling (3 regions) → (batch, 12, 200)
  → Global avg pool → (batch, 200)
  → EEGImageProjection MLP → (batch, 768)  ← CLIP-L pooled dim
  → FLUX.1-schnell (pooled_prompt_embeds override) → 512×512 image
```

**Licenses**: FLUX.1-schnell — Apache 2.0 | CLIP ViT-L/14 — MIT | BCIC-IV-2a — public dataset

In [1]:
# Install required packages not present in the default environment
# diffusers >= 0.30 provides FluxPipeline; protobuf is needed for FLUX's T5 tokenizer
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers>=0.30.0",
    "protobuf",
], check=True)
print("Package installation complete.")

Package installation complete.


In [2]:
import sys
import os
import ssl
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
from timeit import default_timer as timer
import lmdb
import pickle
import copy
import gc
import math
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image

ssl._create_default_https_context = ssl._create_unverified_context

def setup_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(42)
print("Imports done.")

Imports done.


## Configuration

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
CSBRAIN_WEIGHTS_PATH = "pth/CSBrain.pth"
LMDB_DIR             = "data/BCICIV2a/processed_lmdb"
SAVE_DIR             = "pth_downtasks/eeg_image"
CACHE_DIR            = "cache/eeg_image"

# ── Generation models — Apache 2.0, fit in ~8 GB VRAM / ~8 GB system RAM ───
KANDINSKY_PRIOR_MODEL   = "kandinsky-community/kandinsky-2-2-prior"    # Apache 2.0
KANDINSKY_DECODER_MODEL = "kandinsky-community/kandinsky-2-2-decoder"  # Apache 2.0

# ── Architecture ─────────────────────────────────────────────────────────────
NUM_CLASSES     = 4
EEG_DIM         = 200     # CSBrain d_model / pooled output
IMAGE_EMBED_DIM = 1280    # Kandinsky 2.2 prior output dim (CLIP ViT-G/14 → 1280)
PROJ_HIDDEN     = 1024
DROPOUT         = 0.1

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE   = 64
EPOCHS       = 40
LR           = 5e-4
WEIGHT_DECAY = 0.01

# ── Image generation (Kandinsky 2.2) ─────────────────────────────────────────
IMAGE_HEIGHT       = 512
IMAGE_WIDTH        = 512
PRIOR_STEPS        = 25
DECODER_STEPS      = 50
KANDINSKY_GUIDANCE = 4.0

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    free_gb  = torch.cuda.mem_get_info()[0] / 1e9
    total_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f"VRAM   : {free_gb:.1f} / {total_gb:.1f} GB free")

CLASS_NAMES = {0: "Left Hand", 1: "Right Hand", 2: "Both Feet", 3: "Tongue"}

## 1. BCIC-IV-2a Data Loader

Reads from the LMDB already created in `eeg_llm_notebook.ipynb`. Each sample is a preprocessed EEG trial of shape `(22, 4, 200)`.

In [4]:
def to_tensor(array):
    return torch.from_numpy(array).float()


class BCICDataset(Dataset):
    """LMDB-backed BCIC-IV-2a dataset."""
    def __init__(self, data_dir, mode='train'):
        self.db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)
        with self.db.begin(write=False) as txn:
            self.keys = pickle.loads(txn.get('__keys__'.encode()))[mode]
        self.mode = mode

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        with self.db.begin(write=False) as txn:
            pair = pickle.loads(txn.get(key.encode()))
        return to_tensor(pair['sample'] / 100), int(pair['label'])


train_set = BCICDataset(LMDB_DIR, 'train')
val_set   = BCICDataset(LMDB_DIR, 'val')
test_set  = BCICDataset(LMDB_DIR, 'test')

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

sample_eeg, sample_lbl = train_set[0]
print(f"EEG shape: {sample_eeg.shape}, label: {sample_lbl}")

Train: 2784 | Val: 1152 | Test: 1152
EEG shape: torch.Size([22, 4, 200]), label: 0


## 2. Visual Prompt Templates & Class Descriptions

- **Visual prompts** → rich image-generation prompts for Approach 1 (text-guided FLUX generation)
- **Class descriptions** → natural-language EEG descriptions used to extract **CLIP text embedding targets** for training Approach 2

In [5]:
# ── Approach 1: visual prompts for FLUX ────────────────────────────────────
VISUAL_PROMPTS = {
    0: (
        "Scientific medical illustration of a human brain viewed from above. "
        "Right hemisphere motor cortex glowing with electric blue neural activity, "
        "representing left hand motor imagery. Event-related desynchronization shown "
        "as dark suppression regions. White background, high detail, 4K medical diagram."
    ),
    1: (
        "Scientific medical illustration of a human brain viewed from above. "
        "Left hemisphere motor cortex (C3 region) glowing orange-red, representing "
        "right hand motor imagery. Contralateral sensorimotor desynchronization visible. "
        "White background, high detail, 4K medical diagram."
    ),
    2: (
        "Scientific medical illustration of a human brain viewed from above. "
        "Central midline vertex (Cz) glowing bright teal-green symmetrically, "
        "representing bilateral feet motor imagery in the supplementary motor area. "
        "White background, high detail, 4K medical diagram."
    ),
    3: (
        "Scientific medical illustration of a human brain viewed from above. "
        "Bilateral lateral sensorimotor cortex glowing golden-yellow, representing "
        "tongue motor imagery in the orofacial region. Symmetric lateral activation. "
        "White background, high detail, 4K medical diagram."
    ),
}

# ── Approach 2: descriptions for CLIP text embedding targets ─────────────
CLASS_DESCRIPTIONS = {
    0: [
        "Left hand motor imagery. Event-related desynchronization in mu and beta bands over right sensorimotor cortex. Contralateral motor planning.",
        "Imagined left hand movement. Right hemisphere C4 electrode desynchronization. Suppressed mu rhythm over right central region.",
        "EEG left hand motor imagery. Right-lateralized sensorimotor desynchronization 8-30 Hz range.",
    ],
    1: [
        "Right hand motor imagery. Event-related desynchronization in mu and beta bands over left sensorimotor cortex. Contralateral motor planning.",
        "Imagined right hand movement. Left hemisphere C3 electrode desynchronization. Suppressed mu rhythm over left central region.",
        "EEG right hand motor imagery. Left-lateralized sensorimotor desynchronization 8-30 Hz range.",
    ],
    2: [
        "Both feet motor imagery. Bilateral desynchronization over central midline area Cz. Supplementary motor area activation for lower limb imagery.",
        "Imagined foot movement. Mu and beta suppression over vertex Cz. Bilateral foot movement imagery.",
        "EEG feet motor imagery. Midline central desynchronization in sensorimotor bands. Supplementary motor area.",
    ],
    3: [
        "Tongue motor imagery. Event-related desynchronization over lateral sensorimotor regions bilaterally. Orofacial motor imagery.",
        "Imagined tongue movement. Bilateral sensorimotor desynchronization. Face and tongue representation in motor cortex.",
        "EEG tongue motor imagery. Widespread sensorimotor desynchronization bilateral distribution. Lateral primary motor cortex.",
    ],
}

# Style prefix passed to FLUX T5 encoder to guide visual style in Approach 2
STYLE_PROMPT = (
    "Scientific brain MRI visualization, neural activity heatmap, "
    "motor cortex activation, medical illustration, professional diagram, "
    "clean white background, high detail"
)

print(f"Defined {len(VISUAL_PROMPTS)} visual prompts and {len(CLASS_DESCRIPTIONS)} class description sets.")

Defined 4 visual prompts and 4 class description sets.


## 3. Load CSBrain Encoder (Frozen)

Builds CSBrain for the 22-channel BCIC-IV-2a electrode layout, loads pre-trained weights, and freezes all parameters. The output after token-reducing is `(batch, 12, 200)` which is then globally pooled to `(batch, 200)`.

In [6]:
from models.CSBrain import CSBrain


def build_bciciv2a_encoder(weights_path):
    """Build frozen CSBrain for the 22-channel BCIC-IV-2a layout."""
    bci42a_brain_regions = [
        0,
        4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4,
        1, 1, 1, 1,
    ]
    bci42a_electrode_labels = [
        "Fz",
        "FC3", "FC1", "FCZ", "FC2", "FC4",
        "C5", "C3", "C1", "CZ", "C2", "C4", "C6",
        "CP3", "CP1", "CPZ", "CP2", "CP4",
        "P1", "PZ", "P2", "POZ"
    ]
    bci42a_topology = {
        0: ["Fz"],
        4: ["FC3", "FC1", "FCZ", "FC2", "FC4", "C5", "C3", "C1",
            "CZ", "C2", "C4", "C6", "CP3", "CP1", "CPZ", "CP2", "CP4"],
        1: ["P1", "PZ", "P2", "POZ"]
    }
    region_groups = {}
    for i, region in enumerate(bci42a_brain_regions):
        region_groups.setdefault(region, []).append((i, bci42a_electrode_labels[i]))
    sorted_indices = []
    for region in sorted(region_groups.keys()):
        elecs = sorted(region_groups[region],
                       key=lambda x: bci42a_topology[region].index(x[1]))
        sorted_indices.extend([e[0] for e in elecs])

    encoder = CSBrain(
        in_dim=200, out_dim=200, d_model=200,
        dim_feedforward=800, seq_len=30,
        n_layer=12, nhead=8,
        brain_regions=bci42a_brain_regions,
        sorted_indices=sorted_indices,
    )
    if os.path.exists(weights_path):
        sd = torch.load(weights_path, map_location='cpu')
        sd = {k.replace("module.", ""): v for k, v in sd.items()}
        msd = encoder.state_dict()
        msd.update({k: v for k, v in sd.items() if k in msd and v.size() == msd[k].size()})
        encoder.load_state_dict(msd)
        print("Loaded pretrained CSBrain weights.")
    else:
        print("WARNING: No CSBrain weights found — using random init.")

    encoder.proj_out = nn.Identity()   # expose raw 200-dim features
    for p in encoder.parameters():
        p.requires_grad = False
    return encoder, encoder.area_config


class EEGTokenReducer(nn.Module):
    """Pools EEG channels within each brain region → (batch, n_regions, n_patches, d_model)."""
    def __init__(self, area_config):
        super().__init__()
        self.area_config = area_config

    def forward(self, x):
        # x: (batch, n_ch, n_patches, d_model)
        tokens = [x[:, self.area_config[r]['slice'], :, :].mean(1)
                  for r in sorted(self.area_config)]
        return torch.stack(tokens, dim=1)  # (batch, n_regions, n_patches, d_model)


eeg_encoder, area_config = build_bciciv2a_encoder(CSBRAIN_WEIGHTS_PATH)
eeg_encoder = eeg_encoder.to(DEVICE).eval()
token_reducer = EEGTokenReducer(area_config).to(DEVICE)

# Shape check
with torch.no_grad():
    dummy = torch.randn(2, 22, 4, 200).to(DEVICE)
    feats = eeg_encoder(dummy)                          # (2, 22, 4, 200)
    tokens = token_reducer(feats)                       # (2, n_regions, 4, 200)
    pooled = tokens.reshape(2, -1, 200).mean(1)        # (2, 200)

print(f"CSBrain output : {feats.shape}")
print(f"Token reduced  : {tokens.shape}")
print(f"Global pooled  : {pooled.shape}")

Loaded pretrained CSBrain weights.
CSBrain output : torch.Size([2, 22, 4, 200])
Token reduced  : torch.Size([2, 3, 4, 200])
Global pooled  : torch.Size([2, 200])


## 4. Extract CLIP Text Embedding Targets

Encode each class description with **CLIP ViT-L/14** (MIT license) to obtain 768-dim pooled text embeddings. These become the alignment targets for training the EEG projection in Approach 2 — no paired image data is required.

In [7]:
# ── Kandinsky 2.2 Prior — extracts 1024-dim image embeddings from text ──────
# These replace the CLIP text embeddings as training targets.
# The prior is loaded here ONCE, used to extract targets + Approach-1 embeddings,
# then unloaded before training to free VRAM.
from diffusers import KandinskyV22PriorPipeline

kandinsky_targets_path      = os.path.join(CACHE_DIR, "kandinsky_class_targets.pt")
approach1_prior_embeds_path = os.path.join(CACHE_DIR, "approach1_prior_embeds.pt")

if os.path.exists(kandinsky_targets_path) and os.path.exists(approach1_prior_embeds_path):
    class_targets           = torch.load(kandinsky_targets_path,      map_location='cpu')
    approach1_prior_embeds  = torch.load(approach1_prior_embeds_path, map_location='cpu')
    print(f"Loaded Kandinsky targets from cache. Shape: {class_targets.shape}")
else:
    print(f"Loading Kandinsky prior ({KANDINSKY_PRIOR_MODEL}) to extract targets...")
    pipe_prior = KandinskyV22PriorPipeline.from_pretrained(
        KANDINSKY_PRIOR_MODEL, torch_dtype=torch.float16
    ).to(DEVICE)
    pipe_prior.set_progress_bar_config(disable=True)

    # ── Class image embedding targets (used to train EEG projection) ──────
    class_targets = torch.zeros(NUM_CLASSES, IMAGE_EMBED_DIM)
    for class_id, descs in CLASS_DESCRIPTIONS.items():
        all_embs = []
        for desc in descs:
            for seed in range(3):   # average 3 seeds per description for stability
                with torch.no_grad():
                    out = pipe_prior(
                        prompt=desc,
                        guidance_scale=KANDINSKY_GUIDANCE,
                        num_inference_steps=PRIOR_STEPS,
                        generator=torch.Generator(DEVICE).manual_seed(seed),
                    )
                all_embs.append(out.image_embeds.cpu().float())   # (1, 1024)
        class_targets[class_id] = torch.cat(all_embs).mean(0)
        print(f"  Class {class_id} ({CLASS_NAMES[class_id]}): extracted {len(all_embs)} image embeddings")

    # ── Approach 1 embeddings from visual text prompts ────────────────────
    approach1_prior_embeds = {}
    for class_id, prompt in VISUAL_PROMPTS.items():
        with torch.no_grad():
            out = pipe_prior(
                prompt=prompt,
                guidance_scale=KANDINSKY_GUIDANCE,
                num_inference_steps=PRIOR_STEPS,
                generator=torch.Generator(DEVICE).manual_seed(42),
            )
        approach1_prior_embeds[class_id] = {
            "image_embeds":          out.image_embeds.cpu().float(),           # (1, 1024)
            "negative_image_embeds": out.negative_image_embeds.cpu().float(),  # (1, 1024)
        }

    torch.save(class_targets,          kandinsky_targets_path)
    torch.save(approach1_prior_embeds, approach1_prior_embeds_path)

    # Unload prior — projection training only needs the saved tensors
    del pipe_prior
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nExtracted & saved targets. Shape: {class_targets.shape}")

# L2-normalise for cosine alignment loss
class_targets = F.normalize(class_targets.float(), dim=-1)
print("Kandinsky class targets ready (L2-normalised).")

# Preview inter-class cosine similarity in Kandinsky image embedding space
sim = class_targets @ class_targets.T
print("\nInter-class Kandinsky image cosine similarities:")
for i in range(NUM_CLASSES):
    row = "  " + CLASS_NAMES[i].ljust(12) + ": "
    row += "  ".join(f"{sim[i,j]:.3f}" for j in range(NUM_CLASSES))
    print(row)

c:\Users\manoj\Projects\Mtech_Project2\CSBrain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Kandinsky prior (kandinsky-community/kandinsky-2-2-prior) to extract targets...


Loading weights: 100%|██████████| 517/517 [00:02<00:00, 210.81it/s, Materializing param=text_projection.weight]
CLIPTextModelWithProjection LOAD REPORT from: C:\Users\manoj\.cache\huggingface\hub\models--kandinsky-community--kandinsky-2-2-prior\snapshots\9fc51ad5732afc5d031724219d22e6c42179c5a8\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 776/776 [00:01<00:00, 535.42it/s, Materializing param=visual_projection.weight]
CLIPVisionModelWithProjection LOAD REPORT from: C:\Users\manoj\.cache\huggingface\hub\models--kandinsky-community--kandinsky-2-2-prior\snapshots\9fc51ad5732afc5d031724219d22e6c42179c5a8\image_encoder
Key                                  | Status     |  | 
-------------------------------------+

RuntimeError: The expanded size of the tensor (1024) must match the existing size (1280) at non-singleton dimension 0.  Target sizes: [1024].  Tensor sizes: [1280]

## 5. Extract & Cache EEG Features

Pass all splits through the frozen CSBrain encoder once and cache the globally-pooled 200-dim features. This avoids re-running the 12-layer transformer every training epoch.

In [ ]:
def extract_eeg_features(loader, desc="Extracting"):
    """Run CSBrain + token reducer + global pool over a DataLoader."""
    all_feats, all_labels = [], []
    eeg_encoder.eval()
    for eeg, labels in tqdm(loader, desc=desc, mininterval=10):
        eeg = eeg.to(DEVICE)
        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=False):
                feats  = eeg_encoder(eeg.float())         # (B, 22, 4, 200)
            tokens = token_reducer(feats)                 # (B, n_regions, 4, 200)
            pooled = tokens.reshape(eeg.shape[0], -1, 200).mean(1)  # (B, 200)
        all_feats.append(pooled.cpu())
        all_labels.append(labels)
    return torch.cat(all_feats), torch.cat(all_labels)


train_feat_path = os.path.join(CACHE_DIR, "train_eeg_feats.pt")
val_feat_path   = os.path.join(CACHE_DIR, "val_eeg_feats.pt")
test_feat_path  = os.path.join(CACHE_DIR, "test_eeg_feats.pt")

if all(os.path.exists(p) for p in [train_feat_path, val_feat_path, test_feat_path]):
    train_feats, train_labels = torch.load(train_feat_path)
    val_feats,   val_labels   = torch.load(val_feat_path)
    test_feats,  test_labels  = torch.load(test_feat_path)
    print("Loaded EEG features from cache.")
else:
    train_feats, train_labels = extract_eeg_features(train_loader, "Train")
    val_feats,   val_labels   = extract_eeg_features(val_loader,   "Val")
    test_feats,  test_labels  = extract_eeg_features(test_loader,  "Test")
    torch.save((train_feats, train_labels), train_feat_path)
    torch.save((val_feats,   val_labels),   val_feat_path)
    torch.save((test_feats,  test_labels),  test_feat_path)
    print("Extracted and saved EEG features.")

print(f"Shapes — Train: {train_feats.shape} | Val: {val_feats.shape} | Test: {test_feats.shape}")
print(f"Label distribution (train): { {k: (train_labels==k).sum().item() for k in range(NUM_CLASSES)} }")

## 6. EEGImageProjection Model

A 3-layer MLP that maps the 200-dim globally-pooled EEG feature directly to the **1024-dim Kandinsky 2.2 image embedding space** (the output space of the Kandinsky prior network).

At inference the output is passed directly to the **Kandinsky 2.2 decoder** as `image_embeds`, generating a brain-visualization image conditioned on the EEG signal — no text encoder or prior network needed at inference time.

In [ ]:
class EEGImageProjection(nn.Module):
    """
    Projects pooled CSBrain features (200-dim) into Kandinsky 2.2 image embedding
    space (1024-dim). The output is passed to the Kandinsky 2.2 decoder as
    `image_embeds` to generate brain-visualization images conditioned on EEG.

    Training objective: MSE + cosine-similarity loss against Kandinsky image
    embeddings extracted from the prior using class motor imagery descriptions.
    No paired EEG-image data needed.
    """
    def __init__(self, in_dim=200, clip_dim=1024, hidden=1024, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, clip_dim),
            nn.LayerNorm(clip_dim),
        )

    def forward(self, x):  # x: (batch, in_dim)
        return self.net(x)  # (batch, clip_dim)


projection = EEGImageProjection(
    in_dim=EEG_DIM, clip_dim=IMAGE_EMBED_DIM, hidden=PROJ_HIDDEN, dropout=DROPOUT
).to(DEVICE)

n_params = sum(p.numel() for p in projection.parameters())
print(f"EEGImageProjection parameters: {n_params:,}")

with torch.no_grad():
    dummy_feat = torch.randn(4, EEG_DIM).to(DEVICE)
    dummy_out  = projection(dummy_feat)
print(f"Input shape  : {dummy_feat.shape}")
print(f"Output shape : {dummy_out.shape}  (→ Kandinsky 2.2 decoder image_embeds)")

## 7. Train EEG-to-CLIP Alignment

Training the projection with two complementary losses:
- **MSE loss** — exact embedding value regression
- **Cosine similarity loss** — directional alignment in CLIP space

Only the projection MLP is trained. CSBrain remains fully frozen.

In [ ]:
def train_kandinsky_alignment(
    projection, train_feats, train_labels, val_feats, val_labels,
    class_targets, epochs, lr, device, save_dir
):
    optimizer = torch.optim.AdamW(projection.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=5e-6
    )

    feat_ds = TensorDataset(train_feats, train_labels)
    loader  = DataLoader(feat_ds, batch_size=256, shuffle=True, drop_last=False)

    val_ds      = TensorDataset(val_feats, val_labels)
    val_loader_ = DataLoader(val_ds, batch_size=256, shuffle=False)

    class_targets = class_targets.to(device)
    best_val_cos  = -1.0
    best_state    = None
    train_losses, val_coses = [], []

    print(f"{'Epoch':>6}  {'Loss':>8}  {'Val cos':>8}  {'LR':>10}")
    print("-" * 42)

    for epoch in range(epochs):
        projection.train()
        total_loss = 0.0

        for feats, labels in loader:
            feats  = feats.to(device)
            labels = labels.to(device)

            targets = class_targets[labels]          # (B, 1024) normalised targets
            preds   = projection(feats)              # (B, 1024)
            preds_n = F.normalize(preds, dim=-1)

            mse_loss = F.mse_loss(preds, targets)
            cos_loss = (1.0 - F.cosine_similarity(preds_n, targets, dim=-1)).mean()
            loss     = 0.5 * mse_loss + 0.5 * cos_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(projection.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        projection.eval()
        cos_scores = []
        with torch.no_grad():
            for feats, labels in val_loader_:
                preds = F.normalize(projection(feats.to(device)), dim=-1)
                tgts  = class_targets[labels.to(device)]
                cos_scores.append(F.cosine_similarity(preds, tgts, dim=-1).mean().item())
        val_cos = float(np.mean(cos_scores))

        avg_loss = total_loss / len(loader)
        lr_now   = optimizer.param_groups[0]['lr']
        train_losses.append(avg_loss)
        val_coses.append(val_cos)

        if val_cos > best_val_cos:
            best_val_cos = val_cos
            best_state   = copy.deepcopy(projection.state_dict())
            os.makedirs(save_dir, exist_ok=True)
            torch.save(best_state, os.path.join(save_dir, "best_eeg_image_projection.pth"))
            marker = " *"
        else:
            marker = ""

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"{epoch+1:>6}  {avg_loss:>8.4f}  {val_cos:>8.4f}  {lr_now:>10.2e}{marker}")

    print(f"\nBest val cosine similarity (Kandinsky space): {best_val_cos:.4f}")
    return best_state, train_losses, val_coses


print("Starting Kandinsky image embedding alignment training...")
best_state, train_losses, val_coses = train_kandinsky_alignment(
    projection, train_feats, train_labels, val_feats, val_labels,
    class_targets, epochs=EPOCHS, lr=LR, device=DEVICE, save_dir=SAVE_DIR
)

projection.load_state_dict(best_state)
projection.eval()
print("Loaded best projection weights.")

In [ ]:
# Training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, color='steelblue')
ax1.set_title("Training Loss (MSE + Cosine)"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(val_coses, color='darkorange')
ax2.set_title("Validation Cosine Similarity to CLIP Target")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Cosine Similarity")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curve.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved training curve.")

## 8. Projection Quality Analysis

Inspect how well the EEG-projected embeddings cluster by class in **Kandinsky image embedding space**. If the projection learned meaningful structure, class clusters should be distinguishable from each other.

In [ ]:
projection.eval()
with torch.no_grad():
    test_image_embs = projection(test_feats.to(DEVICE)).cpu().float()  # (N_test, 1024)

# Class-average projected embeddings
class_avg_embs = torch.zeros(NUM_CLASSES, IMAGE_EMBED_DIM)
for c in range(NUM_CLASSES):
    mask = (test_labels == c)
    class_avg_embs[c] = F.normalize(test_image_embs[mask].mean(0, keepdim=True), dim=-1).squeeze()

# Inter-class cosine similarity (projected vs projected)
print("Inter-class cosine similarities (projected EEG embeddings):")
proj_sim = class_avg_embs @ class_avg_embs.T
for i in range(NUM_CLASSES):
    row = "  " + CLASS_NAMES[i].ljust(12) + ": "
    row += "  ".join(f"{proj_sim[i,j]:.3f}" for j in range(NUM_CLASSES))
    print(row)

# Alignment to Kandinsky image embedding targets
print("\nAlignment to Kandinsky targets (EEG proj vs prior image embs):")
align_sim = class_avg_embs @ class_targets.T
for i in range(NUM_CLASSES):
    best_match = CLASS_NAMES[align_sim[i].argmax().item()]
    row = "  " + CLASS_NAMES[i].ljust(12) + ": "
    row += "  ".join(f"{align_sim[i,j]:.3f}" for j in range(NUM_CLASSES))
    row += f"  → best match: {best_match}"
    print(row)

## 9. Load Kandinsky 2.2 Decoder

**License**: [Apache 2.0](https://huggingface.co/kandinsky-community/kandinsky-2-2-decoder)

The **Kandinsky 2.2 decoder** takes a 1024-dim image embedding and generates a 512×512 image. It loads in ~3.5 GB VRAM and requires only ~8 GB system RAM — well within the available budget.

```
Pipeline:
  Approach 1 (text)  → prior image_embeds (saved in cell-11) → decoder → image
  Approach 2 (EEG)   → EEG projection (1024-dim)             → decoder → image
```

The prior (loaded once in cell-11 to extract targets) is no longer needed at inference time.

In [ ]:
# Free large EEG encoder from GPU — projection (~8 MB) stays on GPU
eeg_encoder.cpu()
token_reducer.cpu()
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    free_gb  = torch.cuda.mem_get_info()[0] / 1e9
    total_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f"VRAM before decoder load: {free_gb:.1f} / {total_gb:.1f} GB free")

from diffusers import KandinskyV22Pipeline

print(f"\nLoading {KANDINSKY_DECODER_MODEL} (Apache 2.0)...")
pipe_decoder = KandinskyV22Pipeline.from_pretrained(
    KANDINSKY_DECODER_MODEL,
    torch_dtype=torch.float16,
).to(DEVICE)
print("Kandinsky 2.2 decoder loaded on GPU.")

if torch.cuda.is_available():
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM after decoder load : {free_gb:.1f} GB free")

# Neutral negative embedding (zeros = no negative guidance class bias)
ZERO_NEG_EMB = torch.zeros(1, IMAGE_EMBED_DIM, dtype=torch.float16, device=DEVICE)

## 10. Approach 1 — Text-Guided Image Generation

Use the Kandinsky prior image embeddings extracted from the visual text prompts (cell-11) to generate one representative brain-visualization per class. The embeddings were computed once with fixed seeds and cached, so generation is purely decoder inference.

In [ ]:
approach1_dir = os.path.join(SAVE_DIR, "approach1_text")
os.makedirs(approach1_dir, exist_ok=True)

approach1_images = {}

for class_id in range(NUM_CLASSES):
    prior_embs = approach1_prior_embeds[class_id]
    img_emb    = prior_embs["image_embeds"].to(dtype=torch.float16, device=DEVICE)          # (1, 1024)
    neg_emb    = prior_embs["negative_image_embeds"].to(dtype=torch.float16, device=DEVICE)  # (1, 1024)

    print(f"  [{class_id}] {CLASS_NAMES[class_id]} ...")

    with torch.no_grad():
        image = pipe_decoder(
            image_embeds=img_emb,
            negative_image_embeds=neg_emb,
            height=IMAGE_HEIGHT,
            width=IMAGE_WIDTH,
            num_inference_steps=DECODER_STEPS,
            guidance_scale=KANDINSKY_GUIDANCE,
            generator=torch.Generator(device=DEVICE).manual_seed(42),
        ).images[0]

    approach1_images[class_id] = image
    fname = f"class_{class_id}_{CLASS_NAMES[class_id].replace(' ', '_')}.png"
    image.save(os.path.join(approach1_dir, fname))
    print(f"    Saved → {fname}")

print("\nApproach 1 done.")

## 11. Approach 2 — EEG-Conditioned Image Generation

The trained EEG projection maps each EEG signal to a **1024-dim Kandinsky image embedding** and passes it directly to the decoder. No prior or text encoder is involved at inference time.

```
EEG → CSBrain → pool → (200,) → EEGImageProjection → (1024,) → Kandinsky decoder → image
```

**Why this works**: the projection is trained to predict the Kandinsky prior's output for the class motor imagery description. At inference the decoder sees an embedding it was trained to decode — just sourced from EEG instead of text.

In [ ]:
# Run EEG projection on all test samples in one GPU pass, cache results on CPU
projection.eval()
with torch.no_grad():
    test_image_embs_all = projection(test_feats.to(DEVICE)).cpu().float()  # (N_test, 1024)

print(f"Projected test EEG features → {test_image_embs_all.shape}  on CPU")

In [ ]:
# ── Class-average EEG-conditioned generation (one image per class) ─────────
approach2_dir = os.path.join(SAVE_DIR, "approach2_eeg")
os.makedirs(approach2_dir, exist_ok=True)

approach2_images = {}

for class_id in range(NUM_CLASSES):
    mask      = (test_labels == class_id)
    class_emb = test_image_embs_all[mask].mean(0, keepdim=True)               # (1, 1024) float32
    class_emb_gpu = class_emb.to(dtype=torch.float16, device=DEVICE)          # GPU float16

    print(f"  [{class_id}] {CLASS_NAMES[class_id]} (n={mask.sum().item()}) ...")

    with torch.no_grad():
        image = pipe_decoder(
            image_embeds=class_emb_gpu,
            negative_image_embeds=ZERO_NEG_EMB,
            height=IMAGE_HEIGHT,
            width=IMAGE_WIDTH,
            num_inference_steps=DECODER_STEPS,
            guidance_scale=KANDINSKY_GUIDANCE,
            generator=torch.Generator(device=DEVICE).manual_seed(42 + class_id),
        ).images[0]

    approach2_images[class_id] = image
    fname = f"class_{class_id}_{CLASS_NAMES[class_id].replace(' ', '_')}_eeg.png"
    image.save(os.path.join(approach2_dir, fname))
    print(f"    Saved → {fname}")

print("\nApproach 2 class-average images done.")

In [ ]:
# ── Per-sample EEG-conditioned generation (first 8 test samples) ──────────
samples_dir = os.path.join(SAVE_DIR, "approach2_samples")
os.makedirs(samples_dir, exist_ok=True)

N_SAMPLES     = 8
sample_images = []
sample_labels = []

for i in range(N_SAMPLES):
    sample_emb_gpu = test_image_embs_all[i].unsqueeze(0).to(dtype=torch.float16, device=DEVICE)
    true_label     = test_labels[i].item()

    with torch.no_grad():
        image = pipe_decoder(
            image_embeds=sample_emb_gpu,
            negative_image_embeds=ZERO_NEG_EMB,
            height=IMAGE_HEIGHT,
            width=IMAGE_WIDTH,
            num_inference_steps=DECODER_STEPS,
            guidance_scale=KANDINSKY_GUIDANCE,
            generator=torch.Generator(device=DEVICE).manual_seed(100 + i),
        ).images[0]

    sample_images.append(image)
    sample_labels.append(true_label)

    fname = f"sample_{i:03d}_class_{CLASS_NAMES[true_label].replace(' ', '_')}.png"
    image.save(os.path.join(samples_dir, fname))
    print(f"  Sample {i:2d} | true: {CLASS_NAMES[true_label]:<12} → saved")

print("\nPer-sample EEG-conditioned images done.")

## 12. Visualization

In [ ]:
# ── 4-class comparison: Approach 1 (text) vs Approach 2 (EEG) ─────────────
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(22, 12))
fig.suptitle(
    "EEG → Image Generation  |  BCIC-IV-2a Motor Imagery  |  FLUX.1-schnell (Apache 2.0)",
    fontsize=15, fontweight='bold', y=1.01
)

row_labels = [
    "Approach 1\n(Visual text prompt → FLUX)",
    "Approach 2\n(EEG → CLIP projection → FLUX)",
]

for row, (images, row_label) in enumerate([(approach1_images, row_labels[0]),
                                            (approach2_images, row_labels[1])]):
    for col in range(NUM_CLASSES):
        ax = axes[row, col]
        ax.imshow(images[col])
        ax.axis('off')
        if row == 0:
            ax.set_title(CLASS_NAMES[col], fontsize=13, fontweight='bold', pad=8)
        if col == 0:
            ax.set_ylabel(row_label, fontsize=10, rotation=90, labelpad=10)

plt.tight_layout()
save_path = os.path.join(SAVE_DIR, "comparison_grid.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Comparison grid saved → {save_path}")

In [ ]:
# ── Per-sample gallery (Approach 2) ────────────────────────────────────────
COLS = 4
ROWS = math.ceil(N_SAMPLES / COLS)

fig, axes = plt.subplots(ROWS, COLS, figsize=(5 * COLS, 5 * ROWS))
axes = axes.flatten()

fig.suptitle(
    "Per-sample EEG-conditioned Generation (Approach 2)\n"
    "Each image is conditioned on a unique EEG trial's projected CLIP embedding",
    fontsize=13, fontweight='bold'
)

for i, (img, lbl) in enumerate(zip(sample_images, sample_labels)):
    axes[i].imshow(img)
    axes[i].set_title(f"Sample {i}\nTrue: {CLASS_NAMES[lbl]}", fontsize=10)
    axes[i].axis('off')

for j in range(len(sample_images), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
save_path = os.path.join(SAVE_DIR, "sample_gallery.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Sample gallery saved → {save_path}")

In [ ]:
# ── EEG signal + generated image side-by-side for first 4 test samples ────
fig = plt.figure(figsize=(20, 10))
fig.suptitle("EEG Signal → Generated Brain Visualization", fontsize=14, fontweight='bold')

N_SHOW = min(4, N_SAMPLES)
gs = GridSpec(N_SHOW, 2, figure=fig, width_ratios=[2, 1], wspace=0.05, hspace=0.4)

for i in range(N_SHOW):
    true_label = sample_labels[i]

    # EEG subplot (show first 4 channels, Cz region)
    ax_eeg = fig.add_subplot(gs[i, 0])
    eeg_raw = test_feats[i].numpy()  # (200,) pooled CSBrain feature
    ax_eeg.plot(eeg_raw, linewidth=0.8, color='steelblue', alpha=0.8)
    ax_eeg.set_title(f"Sample {i}  |  True class: {CLASS_NAMES[true_label]}",
                     fontsize=10, loc='left')
    ax_eeg.set_xlabel("CSBrain pooled feature dim")
    ax_eeg.set_ylabel("Activation")
    ax_eeg.grid(True, alpha=0.3)

    # Generated image subplot
    ax_img = fig.add_subplot(gs[i, 1])
    ax_img.imshow(sample_images[i])
    ax_img.axis('off')
    ax_img.set_title("Generated image", fontsize=9)

save_path = os.path.join(SAVE_DIR, "eeg_to_image_sidebyside.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Side-by-side plot saved → {save_path}")

## 13. Quantitative: Kandinsky-Space Retrieval Accuracy

Nearest-neighbour classification in Kandinsky image embedding space: for each test EEG sample, project it and find the closest class target. This measures whether the EEG projection correctly clusters by class without any generation or text comparison.

In [ ]:
projection.eval()
class_targets_dev = class_targets.to(DEVICE)   # (4, 1024) on GPU

with torch.no_grad():
    proj_embs = F.normalize(projection(test_feats.to(DEVICE)), dim=-1)  # (N_test, 1024) GPU

sims     = proj_embs @ class_targets_dev.T   # (N_test, 4) GPU
pred_ids = sims.argmax(dim=-1).cpu()         # back to CPU for comparison

correct = (pred_ids == test_labels).sum().item()
total   = len(test_labels)
acc     = correct / total

print(f"\nKandinsky-space nearest-neighbour accuracy: {acc:.4f}  ({correct}/{total})")
print(f"Chance level (4 classes)                  : 0.2500")

print("\nPer-class accuracy:")
for c in range(NUM_CLASSES):
    mask    = (test_labels == c)
    c_acc   = (pred_ids[mask] == c).float().mean().item()
    c_count = mask.sum().item()
    print(f"  {CLASS_NAMES[c]:<12}: {c_acc:.4f}  ({int(c_acc*c_count)}/{c_count})")

In [ ]:
try:
    from sklearn.manifold import TSNE
    import matplotlib.cm as cm

    # Use at most 400 samples for speed
    n_plot = min(400, len(test_feats))
    idx    = torch.randperm(len(test_feats))[:n_plot]

    # Run projection on GPU, then move to CPU for sklearn
    with torch.no_grad():
        embs_plot = projection(test_feats[idx].to(DEVICE)).cpu().float().numpy()
    labels_plot = test_labels[idx].numpy()

    tsne    = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    embs_2d = tsne.fit_transform(embs_plot)

    colors = cm.tab10(np.linspace(0, 1, NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(8, 7))
    for c in range(NUM_CLASSES):
        mask = (labels_plot == c)
        ax.scatter(embs_2d[mask, 0], embs_2d[mask, 1],
                   color=colors[c], label=CLASS_NAMES[c], alpha=0.6, s=30, edgecolors='none')
    ax.legend(fontsize=11, markerscale=1.5)
    ax.set_title("t-SNE of EEG→CLIP Projected Embeddings (test set)", fontsize=13)
    ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, "tsne_clip_embeddings.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"t-SNE plot saved → {save_path}")

except ImportError:
    print("scikit-learn not installed — skipping t-SNE. Run: pip install scikit-learn")

## Summary

| Component | Model | License | Role |
|---|---|---|---|
| EEG Encoder | CSBrain (pretrained) | — | Extract 200-dim EEG representations |
| Image Embedding Targets | Kandinsky 2.2 Prior | **Apache 2.0** | Provide 1024-dim image embedding targets for training |
| EEG Projection | 3-layer MLP (ours) | — | Map EEG → Kandinsky image embedding space |
| Image Generator | Kandinsky 2.2 Decoder | **Apache 2.0** | Generate 512×512 images from image embeddings |

**Why Kandinsky 2.2 instead of FLUX.1-schnell:**
- FLUX.1-schnell requires ~24 GB system RAM to load; Kandinsky 2.2 requires only ~8 GB
- Both are Apache 2.0 licensed
- Kandinsky's decoder accepts `image_embeds` directly, making EEG injection clean and native

### Key insights

1. **No paired EEG-image data needed** — The projection is trained against Kandinsky prior outputs for motor imagery text descriptions. The prior acts as a text-to-image-embedding bridge.

2. **Prior-free inference** — At generation time, only the decoder is loaded (~3.5 GB VRAM). The prior is used once during setup to extract targets and Approach-1 embeddings, then unloaded.

3. **Full open-source stack** — Kandinsky 2.2 (Apache 2.0), CSBrain (project code), BCIC-IV-2a (public benchmark). No proprietary models required.